# Validación y métrica de negocio del proyecto ECU 911

El modelo final de la sesión pasada (bosque con `min_samples_leaf=100`) y el segundo mejor (red neuronal de 32 neuronas), evaluados con validación cruzada en dos esquemas, sobre el periodo de entrenamiento (julio de 2021 a junio de 2025):

- **Temporal**, el que corresponde: el modelo predice el futuro, así que cada pliegue entrena con los días anteriores y valida con los siguientes. Los pliegues se cortan por fecha, no por fila, para que un mismo día no quede repartido entre entrenamiento y validación.
- **Al azar** (estratificado, porque solo el 8% es alta demanda), para comparar: mezcla días de todos los años en entrenamiento y en validación.

El periodo de prueba (julio de 2025 en adelante) no se toca hasta el final: ahí se mide una sola vez el modelo elegido, con el umbral que eligió la validación.

La métrica que venía usando es la precisión promedio (AP). La del negocio es el costo de los errores: un día de alta demanda sin detectar cuesta 5, una falsa alarma cuesta 1.

In [1]:
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from pipeline_proyecto import CORTE, cargar_y_dividir, construir_preparacion

# fecha entra en X para cortar los pliegues; la preparacion la descarta.
X_ent, X_prueba, y_ent, y_prueba = cargar_y_dividir(columnas_extra=["fecha"])
X_ent, y_ent = X_ent.reset_index(drop=True), y_ent.reset_index(drop=True)
print("Entrenamiento:", len(X_ent), "filas, de", X_ent["fecha"].min().date(), "a", X_ent["fecha"].max().date())
print("Prueba:", len(X_prueba), "filas, desde", CORTE.date(), "(no se usa hasta el final)")

Entrenamiento: 327264 filas, de 2021-07-01 a 2025-06-30
Prueba: 94976 filas, desde 2025-07-01 (no se usa hasta el final)


## El costo de cada error

- **Falso negativo (5):** un día de alta demanda que el modelo no anticipa. El ECU 911 no refuerza el turno y las emergencias de ese día esperan más: es el error que el proyecto existe para evitar.
- **Falso positivo (1):** una falsa alarma. Se refuerza un turno que no hacía falta: horas extra y personal movido de otras tareas, pero ninguna emergencia sin atender.

La proporción 5 a 1 es una estimación mía, no un dato del ECU 911. La métrica es el costo por cada 1000 cantón-días: comparable entre pliegues de distinto tamaño.

In [2]:
COSTO_FALSO_NEGATIVO = 5
COSTO_FALSO_POSITIVO = 1
UMBRALES = np.round(np.arange(0.02, 0.99, 0.02), 2)


def costo_por_mil(y, alerta):
    _, falsos_positivos, falsos_negativos, _ = confusion_matrix(y, alerta, labels=[0, 1]).ravel()
    return 1000 * (falsos_negativos * COSTO_FALSO_NEGATIVO + falsos_positivos * COSTO_FALSO_POSITIVO) / len(y)


print(f"Referencias en entrenamiento, costo por 1000 cantón-días:")
print(f"  nunca alertar:  {costo_por_mil(y_ent, np.zeros(len(y_ent))):.1f}")
print(f"  alertar siempre: {costo_por_mil(y_ent, np.ones(len(y_ent))):.1f}")

Referencias en entrenamiento, costo por 1000 cantón-días:
  nunca alertar:  404.2
  alertar siempre: 919.2


## Los pliegues

In [3]:
# Temporal: TimeSeriesSplit sobre los dias, no sobre las filas.
dias = np.sort(X_ent["fecha"].unique())
pliegues_temporales = []
for dias_ent, dias_val in TimeSeriesSplit(n_splits=5).split(dias):
    entrena = np.flatnonzero(X_ent["fecha"].isin(dias[dias_ent]))
    valida = np.flatnonzero(X_ent["fecha"].isin(dias[dias_val]))
    pliegues_temporales.append((entrena, valida))
    print(f"temporal: entrena {pd.Timestamp(dias[dias_ent][0]).date()} a {pd.Timestamp(dias[dias_ent][-1]).date()}"
          f"  valida {pd.Timestamp(dias[dias_val][0]).date()} a {pd.Timestamp(dias[dias_val][-1]).date()}")

pliegues_al_azar = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X_ent, y_ent))
ESQUEMAS = {"temporal": pliegues_temporales, "al azar": pliegues_al_azar}

temporal: entrena 2021-07-01 a 2022-03-03  valida 2022-03-04 a 2022-11-01
temporal: entrena 2021-07-01 a 2022-11-01  valida 2022-11-02 a 2023-07-02
temporal: entrena 2021-07-01 a 2023-07-02  valida 2023-07-03 a 2024-03-01
temporal: entrena 2021-07-01 a 2024-03-01  valida 2024-03-02 a 2024-10-30
temporal: entrena 2021-07-01 a 2024-10-30  valida 2024-10-31 a 2025-06-30


## Validación cruzada de los dos modelos

In [4]:
MODELOS = {
    "bosque min_samples_leaf=100": lambda: RandomForestClassifier(
        n_estimators=100, min_samples_leaf=100, class_weight="balanced", n_jobs=-1, random_state=42),
    "red neuronal (32,)": lambda: MLPClassifier(
        hidden_layer_sizes=(32,), early_stopping=True, max_iter=200, random_state=42),
}

# Probabilidades fuera de pliegue: cada fila validada recibe la del modelo que no la vio.
probabilidades = {}
for esquema, pliegues in ESQUEMAS.items():
    for nombre, crear in MODELOS.items():
        prob = np.full(len(y_ent), np.nan)
        inicio = time.perf_counter()
        for entrena, valida in pliegues:
            pipeline = Pipeline([("preparar", construir_preparacion()), ("modelo", crear())])
            pipeline.fit(X_ent.iloc[entrena], y_ent.iloc[entrena])
            prob[valida] = pipeline.predict_proba(X_ent.iloc[valida])[:, 1]
        probabilidades[(esquema, nombre)] = prob
        print(f"{esquema:9s} {nombre:28s} {time.perf_counter() - inicio:6.1f} s")

temporal  bosque min_samples_leaf=100    28.6 s


temporal  red neuronal (32,)             40.2 s


al azar   bosque min_samples_leaf=100    55.8 s


al azar   red neuronal (32,)             54.7 s


## El umbral de cada modelo según el costo

El bosque usa `class_weight="balanced"` y la red no, así que sus probabilidades no están en la misma escala: cada modelo necesita su propio umbral. Lo elijo con las probabilidades fuera de pliegue de la validación temporal, que imita cómo se usaría el modelo.

In [5]:
umbral = {}
for nombre in MODELOS:
    prob = probabilidades[("temporal", nombre)]
    validadas = ~np.isnan(prob)
    costos = {u: costo_por_mil(y_ent[validadas], prob[validadas] >= u) for u in UMBRALES}
    umbral[nombre] = min(costos, key=costos.get)
    print(f"{nombre:28s} umbral {umbral[nombre]:.2f}   costo {costos[umbral[nombre]]:.1f} por 1000 cantón-días")

bosque min_samples_leaf=100  umbral 0.66   costo 349.0 por 1000 cantón-días


red neuronal (32,)           umbral 0.26   costo 352.0 por 1000 cantón-días


## La tabla: promedio, desviación y cada pliegue

In [6]:
filas = []
for (esquema, nombre), prob in probabilidades.items():
    ap, costo = [], []
    for _, valida in ESQUEMAS[esquema]:
        ap.append(average_precision_score(y_ent.iloc[valida], prob[valida]))
        costo.append(costo_por_mil(y_ent.iloc[valida], prob[valida] >= umbral[nombre]))
    for metrica, valores in [("ap", ap), ("costo_por_1000", costo)]:
        filas.append({
            "esquema": esquema, "modelo": nombre, "metrica": metrica,
            "promedio": round(np.mean(valores), 4), "desviacion": round(np.std(valores), 4),
            "peor_pliegue": round(min(valores) if metrica == "ap" else max(valores), 4),
            "pliegues": [round(v, 4) for v in valores],
        })

tabla = pd.DataFrame(filas)
tabla.to_csv("validacion_proyecto.csv", index=False)
print(tabla.to_string(index=False))

 esquema                      modelo        metrica  promedio  desviacion  peor_pliegue                                           pliegues
temporal bosque min_samples_leaf=100             ap    0.1339      0.0282        0.0890            [0.1321, 0.1317, 0.1389, 0.089, 0.1779]
temporal bosque min_samples_leaf=100 costo_por_1000  349.0006     73.6880      465.2227  [465.2227, 398.5523, 328.3914, 287.037, 265.7995]
temporal          red neuronal (32,)             ap    0.1243      0.0261        0.0805           [0.1197, 0.1313, 0.1284, 0.0805, 0.1616]
temporal          red neuronal (32,) costo_por_1000  351.9547     74.1448      471.7629 [471.7629, 398.4421, 329.7509, 286.9084, 272.9093]
 al azar bosque min_samples_leaf=100             ap    0.1945      0.0052        0.1870             [0.187, 0.1971, 0.1896, 0.199, 0.1998]
 al azar bosque min_samples_leaf=100 costo_por_1000  371.9291      1.7038      374.0852 [374.0852, 369.7768, 373.7949, 370.9532, 371.0353]
 al azar          red neuro

## ¿La diferencia entre los dos modelos es mayor que la variación entre pliegues?

Los dos modelos se validaron en los mismos pliegues, así que se pueden comparar pliegue por pliegue.

In [7]:
for esquema in ESQUEMAS:
    for metrica in ["ap", "costo_por_1000"]:
        bosque = tabla.query("esquema == @esquema and metrica == @metrica and modelo == 'bosque min_samples_leaf=100'").iloc[0]
        red = tabla.query("esquema == @esquema and metrica == @metrica and modelo == 'red neuronal (32,)'").iloc[0]
        diferencias = np.array(red["pliegues"]) - np.array(bosque["pliegues"])
        print(f"{esquema:9s} {metrica:15s} red - bosque por pliegue {np.round(diferencias, 4).tolist()}"
              f"  promedio {diferencias.mean():+.4f}   desviacion entre pliegues del bosque {bosque['desviacion']:.4f}")

temporal  ap              red - bosque por pliegue [-0.0124, -0.0004, -0.0105, -0.0085, -0.0163]  promedio -0.0096   desviacion entre pliegues del bosque 0.0282
temporal  costo_por_1000  red - bosque por pliegue [6.5402, -0.1102, 1.3595, -0.1286, 7.1098]  promedio +2.9541   desviacion entre pliegues del bosque 73.6880
al azar   ap              red - bosque por pliegue [-0.0024, -0.0088, 0.0009, -0.0055, -0.0096]  promedio -0.0051   desviacion entre pliegues del bosque 0.0052
al azar   costo_por_1000  red - bosque por pliegue [6.921, 10.4044, 6.8446, 8.846, 8.6322]  promedio +8.3296   desviacion entre pliegues del bosque 1.7038


## El periodo de prueba, una sola vez

Cada modelo se entrena con todo el periodo de entrenamiento y se mide sobre julio de 2025 en adelante, con el umbral que eligió la validación temporal.

In [8]:
print(f"Referencias en prueba: nunca alertar {costo_por_mil(y_prueba, np.zeros(len(y_prueba))):.1f}, "
      f"alertar siempre {costo_por_mil(y_prueba, np.ones(len(y_prueba))):.1f}")
for nombre, crear in MODELOS.items():
    pipeline = Pipeline([("preparar", construir_preparacion()), ("modelo", crear())]).fit(X_ent, y_ent)
    prob = pipeline.predict_proba(X_prueba)[:, 1]
    alerta = prob >= umbral[nombre]
    _, falsos_positivos, falsos_negativos, verdaderos_positivos = confusion_matrix(y_prueba, alerta).ravel()
    print(f"{nombre:28s} AP {average_precision_score(y_prueba, prob):.4f}   costo {costo_por_mil(y_prueba, alerta):6.1f}"
          f"   detecta {verdaderos_positivos} de {verdaderos_positivos + falsos_negativos} días de alta demanda"
          f" ({verdaderos_positivos / (verdaderos_positivos + falsos_negativos):.0%}), {falsos_positivos} falsas alarmas")

Referencias en prueba: nunca alertar 393.4, alertar siempre 921.3


bosque min_samples_leaf=100  AP 0.2327   costo  348.9   detecta 1260 de 7472 días de alta demanda (17%), 2076 falsas alarmas


red neuronal (32,)           AP 0.2425   costo  361.5   detecta 805 de 7472 días de alta demanda (11%), 995 falsas alarmas
